In [1]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# ── Agent Types ───────────────────────────────
class AgentType(Enum):
    PASSIVE    = 1
    NORMAL     = 2
    AGGRESSIVE = 3

TYPE_COEFFICIENTS = {
    AgentType.PASSIVE:    0.3,
    AgentType.NORMAL:     0.7,
    AgentType.AGGRESSIVE: 1.0
}

# ── Plot Style ────────────────────────────────
plt.rcParams.update({
    'font.size':        10,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'grid.linestyle':   '--',
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

COLORS = {
    'orderly':   '#000000',
    'bunch':     '#555555',
    'high_aggr': '#999999',
    'classical': '#CCCCCC',
}
HATCHES = {
    'orderly':   '',
    'bunch':     '///',
    'high_aggr': '...',
    'classical': 'xxx',
}

print("V10 CPU Idle Experiment: READY")
print(f"Mesa version: {mesa.__version__}")

V10 CPU Idle Experiment: READY
Mesa version: 3.5.1


In [2]:
class QueueAgent(mesa.Agent):
    def __init__(self, model, agent_type):
        super().__init__(model)
        self.agent_type = agent_type
        self.type_coeff = TYPE_COEFFICIENTS[agent_type]
        self.urgency = np.random.uniform(0.3, 1.0)
        self.social_inhibition = np.random.uniform(0.2, 0.8)
        self.exited = False
        self.entry_time = None
        self.latency = None
        self.prev_pos = None

    @property
    def behavior_score(self):
        base_score = (self.type_coeff * self.urgency /
                     self.social_inhibition)
        if self.pos:
            neighbors = self.model.grid.get_neighbors(
                self.pos, moore=True,
                include_center=False, radius=2
            )
            density_factor = 1.0 + (len(neighbors) * 0.05)
        else:
            density_factor = 1.0
        return base_score * density_factor

    def step(self):
        if self.exited or self.pos is None:
            return
        if self.entry_time is None:
            self.entry_time = self.model.steps

        # Record position before move
        self.prev_pos = self.pos
        x, y = self.pos
        move_prob = min(self.behavior_score, 1.0)

        if np.random.random() < move_prob:
            new_y = y - 1
            if new_y < 0:
                self.model.grid.remove_agent(self)
                self.exited = True
                self.latency = (self.model.steps -
                                self.entry_time)
                self.model.exited_count += 1
                self.model.record_agent(self)
                # Count as active move
                self.model.moves_this_step += 1
                return
            new_pos = (x, new_y)
            cell_contents = (
                self.model.grid
                .get_cell_list_contents([new_pos])
            )
            max_occupancy = (
                3 if self.agent_type == AgentType.AGGRESSIVE
                else 2 if self.agent_type == AgentType.NORMAL
                else 1
            )
            if len(cell_contents) < max_occupancy:
                self.model.grid.move_agent(self, new_pos)
                self.model.moves_this_step += 1


class BunchQueueModel(mesa.Model):
    def __init__(self, n_agents=100, width=20,
                 height=30, pct_aggressive=0.15,
                 pct_normal=0.25):
        super().__init__()
        self.width = width
        self.height = height
        self.steps = 0
        self.exited_count = 0
        self.total_agents = n_agents
        self.moves_this_step = 0

        # ── CPU Idle Tracking ─────────────────
        self.idle_history = []
        self.utilization_history = []

        # ── Latency Tracking ──────────────────
        self.latencies = {
            AgentType.PASSIVE:    [],
            AgentType.NORMAL:     [],
            AgentType.AGGRESSIVE: []
        }

        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )

        n_aggressive = int(n_agents * pct_aggressive)
        n_normal     = int(n_agents * pct_normal)
        n_passive    = (n_agents - n_aggressive
                        - n_normal)

        agent_types = (
            [AgentType.AGGRESSIVE] * n_aggressive +
            [AgentType.NORMAL]     * n_normal +
            [AgentType.PASSIVE]    * n_passive
        )
        np.random.shuffle(agent_types)

        for agent_type in agent_types:
            agent = QueueAgent(self, agent_type)
            x = np.random.randint(0, width)
            y = np.random.randint(height // 2, height)
            self.grid.place_agent(agent, (x, y))

    def record_agent(self, agent):
        if agent.latency is not None:
            self.latencies[agent.agent_type].append(
                agent.latency
            )

    def step(self):
        self.steps += 1
        self.moves_this_step = 0

        self.agents.shuffle_do("step")

        # ── Calculate CPU idle this step ──────
        active = (self.total_agents -
                  self.exited_count)
        if active > 0:
            utilization = (self.moves_this_step /
                          active)
            idle_rate = 1.0 - min(utilization, 1.0)
        else:
            utilization = 1.0
            idle_rate = 0.0

        self.utilization_history.append(utilization)
        self.idle_history.append(idle_rate)

    def run(self, max_steps=500):
        for _ in range(max_steps):
            self.step()
            if self.exited_count >= self.total_agents:
                break
        return self.exited_count

    def get_idle_stats(self):
        if not self.idle_history:
            return 0, 0, 0
        avg_idle   = np.mean(self.idle_history)
        peak_idle  = np.max(self.idle_history)
        avg_util   = np.mean(self.utilization_history)
        return avg_idle, peak_idle, avg_util


print("Model with CPU idle tracking defined!")
print("New metrics:")
print("  avg_idle_rate  — average idle across all steps")
print("  peak_idle_rate — worst case idle step")
print("  avg_utilization — average resource utilization")

Model with CPU idle tracking defined!
New metrics:
  avg_idle_rate  — average idle across all steps
  peak_idle_rate — worst case idle step
  avg_utilization — average resource utilization


In [ ]:
# ── V10 PART 1 — Small Scale CPU Idle ────────
# Replicates V1 scenarios with idle tracking
# 100 agents, 20 runs per scenario

print("V10 PART 1 — Small Scale CPU Idle Tracking")
print("="*60)
print("Scale: 100 agents, 20 runs per scenario")
print("="*60)

scenarios_v1 = [
    {'name': 'Orderly\n5% aggr',
     'pct_agg': 0.05, 'pct_norm': 0.20},
    {'name': 'Bunch\n15% aggr',
     'pct_agg': 0.15, 'pct_norm': 0.25},
    {'name': 'High Aggr\n30% aggr',
     'pct_agg': 0.30, 'pct_norm': 0.25},
]

n_runs = 20
results_v1 = []

for scenario in scenarios_v1:
    throughputs  = []
    passive_lats = []
    avg_idles    = []
    peak_idles   = []
    avg_utils    = []

    for run in range(n_runs):
        model = BunchQueueModel(
            n_agents=100,
            width=20, height=30,
            pct_aggressive=scenario['pct_agg'],
            pct_normal=scenario['pct_norm']
        )
        cleared = model.run(max_steps=500)
        avg_idle, peak_idle, avg_util = (
            model.get_idle_stats()
        )

        throughputs.append(cleared)
        avg_idles.append(avg_idle)
        peak_idles.append(peak_idle)
        avg_utils.append(avg_util)

        if model.latencies[AgentType.PASSIVE]:
            passive_lats.extend(
                model.latencies[AgentType.PASSIVE]
            )

    r = {
        'name':       scenario['name'],
        'throughput': np.mean(throughputs),
        'passive_lat':np.mean(passive_lats)
                      if passive_lats else 0,
        'avg_idle':   np.mean(avg_idles),
        'peak_idle':  np.mean(peak_idles),
        'avg_util':   np.mean(avg_utils),
    }
    results_v1.append(r)

    print(f"\nScenario: {scenario['name'].replace(chr(10),' ')}")
    print(f"  Throughput:       {r['throughput']:.1f} agents cleared")
    print(f"  Passive latency:  {r['passive_lat']:.1f} steps")
    print(f"  Avg idle rate:    {r['avg_idle']*100:.1f}%")
    print(f"  Peak idle rate:   {r['peak_idle']*100:.1f}%")
    print(f"  Avg utilization:  {r['avg_util']*100:.1f}%")

print("\n" + "="*60)
print("SMALL SCALE SUMMARY")
print(f"{'Scenario':<16} {'Throughput':>10} "
      f"{'Pass Lat':>10} {'Avg Idle':>10} "
      f"{'Avg Util':>10}")
print("-"*60)
for r in results_v1:
    name = r['name'].replace('\n', ' ')
    print(f"{name:<16} "
          f"{r['throughput']:>10.1f} "
          f"{r['passive_lat']:>10.1f} "
          f"{r['avg_idle']*100:>9.1f}% "
          f"{r['avg_util']*100:>9.1f}%")
print("="*60)

In [ ]:
# ── V10 PART 2 — Scale CPU Idle Experiment ───
# Four aggression levels including near-optimal
# from V6 sweep (75-80% at small scale)

print("V10 PART 2 — Scale CPU Idle Experiment")
print("="*60)
print("Testing CPU idle across scales — 4 scenarios")
print("="*60)

scale_configs = [
    {'name': '100',  'n': 100,  'w': 20,
     'h': 30,  'steps': 500},
    {'name': '500',  'n': 500,  'w': 45,
     'h': 65,  'steps': 1000},
    {'name': '1000', 'n': 1000, 'w': 65,
     'h': 90,  'steps': 1500},
    {'name': '2500', 'n': 2500, 'w': 100,
     'h': 140, 'steps': 2000},
    {'name': '5000', 'n': 5000, 'w': 140,
     'h': 200, 'steps': 3000},
]

aggr_scenarios = [
    {'name': 'Orderly 5%',  'pct_agg': 0.05,
     'pct_norm': 0.20},
    {'name': 'Bunch 15%',   'pct_agg': 0.15,
     'pct_norm': 0.25},
    {'name': 'High 30%',    'pct_agg': 0.30,
     'pct_norm': 0.25},
    {'name': 'Near-Opt 75%','pct_agg': 0.75,
     'pct_norm': 0.20},
]

n_runs = 20
idle_results = {}

for scale in scale_configs:
    print(f"\nScale: {scale['name']} agents")
    print("-"*60)
    idle_results[scale['name']] = {}

    for scenario in aggr_scenarios:
        passive_lats = []
        avg_idles    = []
        avg_utils    = []

        for run in range(n_runs):
            model = BunchQueueModel(
                n_agents=scale['n'],
                width=scale['w'],
                height=scale['h'],
                pct_aggressive=scenario['pct_agg'],
                pct_normal=scenario['pct_norm']
            )
            model.run(max_steps=scale['steps'])
            avg_idle, peak_idle, avg_util = (
                model.get_idle_stats()
            )
            avg_idles.append(avg_idle)
            avg_utils.append(avg_util)

            if model.latencies[AgentType.PASSIVE]:
                passive_lats.extend(
                    model.latencies[AgentType.PASSIVE]
                )

        avg_passive = (np.mean(passive_lats)
                      if passive_lats else 0)
        mean_idle   = np.mean(avg_idles)
        mean_util   = np.mean(avg_utils)

        idle_results[scale['name']][scenario['name']] = {
            'passive':  avg_passive,
            'avg_idle': mean_idle,
            'avg_util': mean_util,
        }

        print(f"  {scenario['name']:<14} "
              f"Passive: {avg_passive:7.1f}  "
              f"Idle: {mean_idle*100:5.1f}%  "
              f"Util: {mean_util*100:5.1f}%")

# ── Summary Table ─────────────────────────────
print("\n" + "="*70)
print("SCALE SUMMARY — CPU IDLE vs AGGRESSION LEVEL")
print("="*70)
print(f"{'Scale':<8} {'Ord 5%':>9} "
      f"{'Bunch 15%':>10} {'High 30%':>10} "
      f"{'NearOpt 75%':>12} {'Best Idle':>10}")
print("-"*70)

for scale in scale_configs:
    r = idle_results[scale['name']]
    orderly  = r['Orderly 5%']['avg_idle']  * 100
    bunch    = r['Bunch 15%']['avg_idle']   * 100
    high     = r['High 30%']['avg_idle']    * 100
    nearopt  = r['Near-Opt 75%']['avg_idle']* 100

    # Find which scenario has lowest idle
    idles = {
        'Ord':    orderly,
        'Bunch':  bunch,
        'High':   high,
        'NearOpt':nearopt
    }
    best = min(idles, key=idles.get)

    print(f"{scale['name']:<8} "
          f"{orderly:>8.1f}% "
          f"{bunch:>9.1f}% "
          f"{high:>9.1f}% "
          f"{nearopt:>11.1f}% "
          f"{best:>10}")

print("="*70)
print("\nBest Idle = scenario with lowest CPU idle rate")
print("(lowest idle = highest utilization = most efficient)")

# ── Passive Latency Summary ───────────────────
print("\n" + "="*70)
print("PASSIVE AGENT LATENCY vs AGGRESSION LEVEL")
print("="*70)
print(f"{'Scale':<8} {'Ord 5%':>9} "
      f"{'Bunch 15%':>10} {'High 30%':>10} "
      f"{'NearOpt 75%':>12} {'Best Lat':>10}")
print("-"*70)

for scale in scale_configs:
    r = idle_results[scale['name']]
    orderly  = r['Orderly 5%']['passive']
    bunch    = r['Bunch 15%']['passive']
    high     = r['High 30%']['passive']
    nearopt  = r['Near-Opt 75%']['passive']

    lats = {
        'Ord':    orderly,
        'Bunch':  bunch,
        'High':   high,
        'NearOpt':nearopt
    }
    best = min(lats, key=lats.get)

    print(f"{scale['name']:<8} "
          f"{orderly:>9.1f} "
          f"{bunch:>10.1f} "
          f"{high:>10.1f} "
          f"{nearopt:>12.1f} "
          f"{best:>10}")

print("="*70)
print("\nAll latency values in simulation steps")
print("Lower = better for passive agents")
print("\nRunning — large scales will take 2-3 hours...")

In [ ]:
# ── V11 THRASHING BOUNDARY ────────────────────
# Find where performance peaks and reverses
# 100 agents only — targeted and fast
# Testing 75% through 95% aggression

print("V11 THRASHING BOUNDARY EXPERIMENT")
print("="*60)
print("Scale: 100 agents, 20 runs per scenario")
print("Finding peak performance and thrashing onset")
print("="*60)

thrash_scenarios = [
    {'pct_agg': 0.60, 'name': '60%'},
    {'pct_agg': 0.65, 'name': '65%'},
    {'pct_agg': 0.70, 'name': '70%'},
    {'pct_agg': 0.75, 'name': '75%'},
    {'pct_agg': 0.80, 'name': '80%'},
    {'pct_agg': 0.85, 'name': '85%'},
    {'pct_agg': 0.90, 'name': '90%'},
    {'pct_agg': 0.95, 'name': '95%'},
]

n_runs = 20
thrash_results = []

for scenario in thrash_scenarios:
    passive_lats = []
    aggr_lats    = []
    avg_idles    = []
    avg_utils    = []
    throughputs  = []

    # Keep normal% reasonable — 
    # cap at remaining capacity
    pct_agg  = scenario['pct_agg']
    pct_norm = min(0.20, 1.0 - pct_agg - 0.05)

    for run in range(n_runs):
        model = BunchQueueModel(
            n_agents=100,
            width=20, height=30,
            pct_aggressive=pct_agg,
            pct_normal=pct_norm
        )
        cleared = model.run(max_steps=500)
        avg_idle, peak_idle, avg_util = (
            model.get_idle_stats()
        )

        throughputs.append(cleared)
        avg_idles.append(avg_idle)
        avg_utils.append(avg_util)

        if model.latencies[AgentType.PASSIVE]:
            passive_lats.extend(
                model.latencies[AgentType.PASSIVE]
            )
        if model.latencies[AgentType.AGGRESSIVE]:
            aggr_lats.extend(
                model.latencies[AgentType.AGGRESSIVE]
            )

    avg_passive = (np.mean(passive_lats)
                  if passive_lats else 0)
    avg_aggr    = (np.mean(aggr_lats)
                  if aggr_lats else 0)
    mean_idle   = np.mean(avg_idles)
    mean_util   = np.mean(avg_utils)
    mean_thru   = np.mean(throughputs)

    thrash_results.append({
        'name':       scenario['name'],
        'pct_agg':    pct_agg,
        'passive':    avg_passive,
        'aggressive': avg_aggr,
        'idle':       mean_idle,
        'util':       mean_util,
        'throughput': mean_thru,
    })

    print(f"  {scenario['name']}  "
          f"Passive: {avg_passive:6.1f}  "
          f"Aggr: {avg_aggr:6.1f}  "
          f"Idle: {mean_idle*100:5.1f}%  "
          f"Util: {mean_util*100:5.1f}%")

# ── Summary ───────────────────────────────────
print("\n" + "="*70)
print("THRASHING BOUNDARY SUMMARY — 100 Agents")
print("="*70)
print(f"{'Aggr%':<8} {'Pass Lat':>9} {'Aggr Lat':>9} "
      f"{'Idle':>8} {'Util':>8} {'vs 75%':>10} "
      f"{'Regime'}")
print("-"*70)

# Find 75% baseline
baseline = next(
    r for r in thrash_results
    if r['name'] == '75%'
)

for r in thrash_results:
    diff = r['passive'] - baseline['passive']
    diff_str = (f"+{diff:.1f}" if diff > 0
                else f"{diff:.1f}")

    # Classify regime
    if r['pct_agg'] < 0.70:
        regime = "Building"
    elif r['pct_agg'] <= 0.80:
        regime = "Optimal zone"
    elif r['passive'] > baseline['passive'] * 1.05:
        regime = "THRASHING"
    else:
        regime = "Near optimal"

    print(f"{r['name']:<8} "
          f"{r['passive']:>9.1f} "
          f"{r['aggressive']:>9.1f} "
          f"{r['idle']*100:>7.1f}% "
          f"{r['util']*100:>7.1f}% "
          f"{diff_str:>10} "
          f"{regime}")

print("="*70)
print("\nvs 75% = passive latency change from 75% baseline")
print("Positive = worse than 75% (thrashing)")
print("Negative = better than 75% (still improving)")

# ── Find optimal point ────────────────────────
best = min(thrash_results,
           key=lambda x: x['passive'])
print(f"\nOptimal aggression: {best['name']}")
print(f"Best passive latency: {best['passive']:.1f} steps")
print(f"Best CPU idle: {best['idle']*100:.1f}%")
print(f"Best utilization: {best['util']*100:.1f}%")

In [3]:
# ── V11 CLEAN RUN — With Tee Logger ──────────
# 10 aggression levels: 5% to 95%
# 5 scales: 100 to 5000 agents
# 60 runs per scenario
# Auto-saves results after each scale
# Logs all output to file automatically

import json
import time
import sys

class Tee:
    def __init__(self, filename):
        self.file = open(filename, 'w')
        self.stdout = sys.stdout
    def write(self, text):
        self.file.write(text)
        self.stdout.write(text)
        self.file.flush()
    def flush(self):
        self.file.flush()
        self.stdout.flush()

sys.stdout = Tee('/home/jc/v11_clean_output.txt')

print("V11 CLEAN RUN — 100% CPU, No Interference")
print("="*60)
print("Running clean — do not interrupt")
print("60 runs per scenario")
print("10 aggression levels: 5% to 95%")
print("5 scales: 100 to 5000 agents")
print("Auto-saving after each scale")
print("="*60)

start_time = time.time()

all_scenarios = [
    {'pct_agg': 0.05, 'name': 'Ord 5%'},
    {'pct_agg': 0.15, 'name': 'Bunch 15%'},
    {'pct_agg': 0.30, 'name': 'High 30%'},
    {'pct_agg': 0.60, 'name': '60%'},
    {'pct_agg': 0.65, 'name': '65%'},
    {'pct_agg': 0.75, 'name': '75%'},
    {'pct_agg': 0.80, 'name': '80%'},
    {'pct_agg': 0.85, 'name': '85%'},
    {'pct_agg': 0.90, 'name': '90%'},
    {'pct_agg': 0.95, 'name': '95%'},
]

scale_configs = [
    {'name': '100',  'n': 100,  'w': 20,
     'h': 30,  'steps': 500},
    {'name': '500',  'n': 500,  'w': 45,
     'h': 65,  'steps': 1000},
    {'name': '1000', 'n': 1000, 'w': 65,
     'h': 90,  'steps': 1500},
    {'name': '2500', 'n': 2500, 'w': 100,
     'h': 140, 'steps': 2000},
    {'name': '5000', 'n': 5000, 'w': 140,
     'h': 200, 'steps': 3000},
]

n_runs = 60
clean_results = {}

for scale in scale_configs:
    print(f"\nScale: {scale['name']} agents")
    print("-"*60)
    clean_results[scale['name']] = {}

    for scenario in all_scenarios:
        passive_lats      = []
        aggr_lats         = []
        avg_idles         = []
        avg_utils         = []
        throughputs       = []
        run_passive_means = []

        pct_agg  = scenario['pct_agg']
        pct_norm = min(0.20, 1.0 - pct_agg - 0.05)

        for run in range(n_runs):
            model = BunchQueueModel(
                n_agents=scale['n'],
                width=scale['w'],
                height=scale['h'],
                pct_aggressive=pct_agg,
                pct_normal=pct_norm
            )
            cleared = model.run(
                max_steps=scale['steps']
            )
            avg_idle, peak_idle, avg_util = (
                model.get_idle_stats()
            )

            throughputs.append(cleared)
            avg_idles.append(avg_idle)
            avg_utils.append(avg_util)

            if model.latencies[AgentType.PASSIVE]:
                run_mean = np.mean(
                    model.latencies[AgentType.PASSIVE]
                )
                run_passive_means.append(run_mean)
                passive_lats.extend(
                    model.latencies[AgentType.PASSIVE]
                )
            if model.latencies[AgentType.AGGRESSIVE]:
                aggr_lats.extend(
                    model.latencies[AgentType.AGGRESSIVE]
                )

        avg_passive = (np.mean(passive_lats)
                      if passive_lats else 0)
        std_passive = (np.std(run_passive_means)
                      if run_passive_means else 0)
        avg_aggr    = (np.mean(aggr_lats)
                      if aggr_lats else 0)
        mean_idle   = np.mean(avg_idles)
        mean_util   = np.mean(avg_utils)
        mean_thru   = np.mean(throughputs)

        clean_results[scale['name']][
            scenario['name']
        ] = {
            'passive':    avg_passive,
            'std':        std_passive,
            'aggressive': avg_aggr,
            'idle':       mean_idle,
            'util':       mean_util,
            'throughput': mean_thru,
        }

        elapsed = (time.time() - start_time) / 60
        print(f"  {scenario['name']:<12} "
              f"Passive: {avg_passive:7.1f} "
              f"(±{std_passive:.1f})  "
              f"Idle: {mean_idle*100:5.1f}%  "
              f"Util: {mean_util*100:5.1f}%  "
              f"[{elapsed:.0f}min]")

    # ── Auto-save after each scale ────────────
    with open('/home/jc/v11_clean_results.json',
              'w') as f:
        json.dump(clean_results, f, indent=2)
    elapsed = (time.time() - start_time) / 60
    print(f"\n  Scale {scale['name']} complete "
          f"— results saved [{elapsed:.0f}min]")

# ── Final Summary ─────────────────────────────
elapsed_total = (time.time() - start_time) / 60
print(f"\n{'='*70}")
print(f"CLEAN RUN COMPLETE")
print(f"Total runtime: {elapsed_total:.0f} minutes")
print(f"{'='*70}")

print(f"\nPASSIVE LATENCY SUMMARY (simulation steps)")
print(f"{'Scale':<8}", end="")
for s in all_scenarios:
    print(f" {s['name']:>10}", end="")
print(f" {'Best':>8}")
print("-"*100)

for scale in scale_configs:
    print(f"{scale['name']:<8}", end="")
    best_lat  = float('inf')
    best_name = ""
    for s in all_scenarios:
        r = clean_results[scale['name']][s['name']]
        print(f" {r['passive']:>10.1f}", end="")
        if r['passive'] < best_lat:
            best_lat  = r['passive']
            best_name = s['name']
    print(f" {best_name:>8}")

print(f"\nCPU IDLE SUMMARY (%)")
print(f"{'Scale':<8}", end="")
for s in all_scenarios:
    print(f" {s['name']:>10}", end="")
print(f" {'Best':>8}")
print("-"*100)

for scale in scale_configs:
    print(f"{scale['name']:<8}", end="")
    best_idle = float('inf')
    best_name = ""
    for s in all_scenarios:
        r = clean_results[scale['name']][s['name']]
        print(f" {r['idle']*100:>9.1f}%", end="")
        if r['idle'] < best_idle:
            best_idle = r['idle']
            best_name = s['name']
    print(f" {best_name:>8}")

print(f"\nSTANDARD DEVIATION SUMMARY (steps)")
print(f"{'Scale':<8}", end="")
for s in all_scenarios:
    print(f" {s['name']:>10}", end="")
print("-"*100)

for scale in scale_configs:
    print(f"{scale['name']:<8}", end="")
    for s in all_scenarios:
        r = clean_results[scale['name']][s['name']]
        print(f" {r['std']:>10.1f}", end="")
    print()

print(f"\nFiles saved:")
print(f"  /home/jc/v11_clean_output.txt")
print(f"  /home/jc/v11_clean_results.json")
print(f"\nSCP to Windows when complete:")
print(f"  scp -P 2222 jc@127.0.0.1:"
      f"/home/jc/v11_clean_output.txt "
      f"C:\\Users\\jcurr\\Desktop\\")
print(f"  scp -P 2222 jc@127.0.0.1:"
      f"/home/jc/v11_clean_results.json "
      f"C:\\Users\\jcurr\\Desktop\\")

V11 CLEAN RUN — 100% CPU, No Interference
Running clean — do not interrupt
60 runs per scenario
10 aggression levels: 5% to 95%
5 scales: 100 to 5000 agents
Auto-saving after each scale

Scale: 100 agents
------------------------------------------------------------
  Ord 5%       Passive:   150.1 (±13.6)  Idle:  73.0%  Util:  27.1%  in]
  Bunch 15%    Passive:   151.6 (±12.9)  Idle:  72.9%  Util:  27.2%  in]
  High 30%     Passive:   145.0 (±16.3)  Idle:  69.0%  Util:  31.2%  in]
  60%          Passive:   138.8 (±17.9)  Idle:  63.1%  Util:  37.2%  in]
  65%          Passive:   135.2 (±22.5)  Idle:  60.8%  Util:  39.3%  in]
  75%          Passive:   129.8 (±34.0)  Idle:  49.6%  Util:  51.0%  in]
  80%          Passive:   130.8 (±38.7)  Idle:  52.4%  Util:  48.4%  in]
  85%          Passive:   130.5 (±42.6)  Idle:  47.6%  Util:  53.5%  in]
  90%          Passive:   131.6 (±34.4)  Idle:  51.5%  Util:  49.2%  in]
  95%          Passive:   127.6 (±33.8)  Idle:  49.4%  Util:  51.7%  in]

  S